## Día 2 · Limpiar datos financieros con IA sin perder el control

Notebook que acompaña al post del reto *IA aplicada a finanzas*. Toma `estados_sucio.csv`
(los datos del día 1 con errores sembrados a propósito por `crear_dataset_sucio.py`), lo
diagnostica, lo limpia dejando registro de cada cambio y valida el resultado contra los
datos originales.

In [1]:
#| label: tbl-crudo
import re
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")
CUENTAS = ["ingresos", "utilidad_neta", "activos", "patrimonio",
           "activo_corriente", "pasivo_corriente", "pasivo_total"]

# Se lee todo como texto: así nada se convierte en silencio antes de revisarlo
crudo = pd.read_csv(DATA / "estados_sucio.csv", dtype=str, keep_default_na=False)
print(f"{len(crudo)} filas, {crudo.shape[1]} columnas")
crudo.head(6)

17 filas, 11 columnas


,Empresa,Fecha de cierre,Ingresos (MM USD),Utilidad Neta,Activos totales,Patrimonio,Activo Corriente,pasivo corriente,Pasivo Total,Moneda,Margen neto
0,walmart,"Jan 31, 2025","$680,985","$19,436","$260,823","$91,013","$79,458","$96,584","$163,402",Dólares,0.029
1,WALMART,31/01/2023,611289.0,11680.0,243197.0,76693.0,75655.0,92198.0,N/A,usd,
2,Target,31/01/2026,"$104,780","$3,705","$59,490","$16,165","$20,005","$21,230","$43,325",USD,0.035
3,target corp.,"Feb 01, 2025",106566,4091,57769,14666,19454,20799,43103,Dólares,3.8%
4,Target Corporation,2022-01-29,"$106,005","$6,946","$53,811","$12,827","$21,573","$21,747","$40,984",USD,0.066
5,Costco,09/03/2023,242.290,6.292,68.994,25.058,35.879,33.583,43.936,US$,"2,6 %"


## 1. Diagnóstico: un perfil que se le puede pasar a la IA

En lugar de pegar el archivo completo en un asistente, se genera un **perfil**: para cada
columna, cuántos valores distintos tiene, qué "formas" toman sus valores y algunos
ejemplos. Es más corto, revela los problemas igual de bien y no expone los datos
completos, algo que importa cuando el archivo es confidencial.

In [2]:
#| label: tbl-perfil
def forma(valor):
    """Reduce un valor a su patrón: dígitos -> 9, letras -> a. '$1,234' -> '$9,999'."""
    return re.sub(r"[A-Za-zÀ-ÿ]+", "a", re.sub(r"\d", "9", valor)) if valor.strip() else "(vacío)"

perfil = pd.DataFrame([{
    "Columna": repr(c),
    "Distintos": crudo[c].nunique(),
    "Formas distintas": crudo[c].map(forma).nunique(),
    "Ejemplos de formas": " · ".join(crudo[c].map(forma).value_counts().index[:4]),
} for c in crudo.columns])
perfil.style.hide(axis="index")

Columna,Distintos,Formas distintas,Ejemplos de formas
'Empresa',15,7,a · a a · a a. · a a a.
'Fecha de cierre',15,4,"99/99/9999 · 9999-99-99 99:99:99 · a 99, 9999 · 9999-99-99"
'Ingresos (MM USD)',15,5,"$999,999 · 999999 · 999.999 · 999999.9"
'Utilidad Neta',15,8,"$9,999 · 9999 · 9.999 · 99.999"
' Activos totales',15,8,"$99,999 · 99999 · 99.999 · 999.999"
'Patrimonio',15,4,"$99,999 · 99999 · 99.999 · 99999.9"
'Activo Corriente',15,5,"$99,999 · 99999 · 99.999 · 99999.9"
'pasivo corriente',15,6,"99999 · $99,999 · 99999.9 · 99.999"
'Pasivo Total',15,8,"$99,999 · 99999 · 99.999 · 999.999"
'Moneda',4,2,a · a$


In [3]:
#| label: out-duplicados
print("Filas idénticas repetidas:", crudo.duplicated().sum())
print("Variantes de nombre de empresa:", sorted(crudo["Empresa"].unique()))

Filas idénticas repetidas: 1
Variantes de nombre de empresa: ['COSTCO WHOLESALE CORP /NEW', 'Costco', 'Costco Wholesale', 'Costco Wholesale Corp.', 'TARGET CORP', 'Target', 'Target Corp', 'Target Corporation', 'WALMART', 'Wal-Mart', 'Walmart', 'Walmart Inc.', 'costco', 'target corp.', 'walmart ']


## 2. Limpieza, una regla por problema

Cada función resuelve un tipo de problema y **deja anotado lo que cambió** en un
registro. La regla de oro: nada se corrige en silencio.

In [4]:
registro = []

def anotar(fila, columna, antes, despues, regla):
    registro.append({"fila": fila, "columna": columna, "antes": antes,
                     "despues": despues, "regla": regla})

# 2.1 Encabezados: sin espacios sobrantes y con nombres estables
ENCABEZADOS = {"empresa": "empresa", "fecha de cierre": "cierre_fiscal",
               "ingresos (mm usd)": "ingresos", "utilidad neta": "utilidad_neta",
               "activos totales": "activos", "patrimonio": "patrimonio",
               "activo corriente": "activo_corriente", "pasivo corriente": "pasivo_corriente",
               "pasivo total": "pasivo_total", "moneda": "moneda", "margen neto": "margen_reportado"}
df = crudo.rename(columns=lambda c: ENCABEZADOS[c.strip().lower()]).copy()

**Nombres de empresa.** Se usa un diccionario explícito y no una coincidencia "difusa":
con tres empresas es fácil de revisar, y una coincidencia aproximada podría confundir
nombres parecidos de empresas distintas. Si aparece un nombre que no está en el
diccionario, el código se detiene en vez de adivinar.

In [5]:
def clave(nombre):
    """walmart , Wal-Mart, WALMART INC. -> walmart"""
    n = re.sub(r"[^a-z ]", " ", nombre.lower())
    n = re.sub(r"\b(inc|corp|corporation|wholesale|new|stores)\b", " ", n)
    return re.sub(r"\s+", "", n)

EMPRESAS = {"walmart": "Walmart", "target": "Target", "costco": "Costco"}
for i, v in df["empresa"].items():
    k = clave(v)
    if k not in EMPRESAS:
        raise ValueError(f"Nombre no reconocido en la fila {i}: {v!r}")
    if v != EMPRESAS[k]:
        anotar(i, "empresa", v, EMPRESAS[k], "nombre canónico")
    df.at[i, "empresa"] = EMPRESAS[k]

**Fechas.** Hay cinco formatos mezclados, y dos de ellos chocan: `09/03/2023` puede ser
9 de marzo (día/mes) o 3 de septiembre (mes/día). Ningún algoritmo genérico puede
resolverlo, pero el conocimiento del negocio sí: Costco cierra a fines de agosto o
inicios de septiembre, nunca en marzo. Cuando ambas lecturas son válidas, gana la que
cae en los meses de cierre conocidos de la empresa.

In [6]:
FORMATOS = ["%Y-%m-%d", "%Y-%m-%d %H:%M:%S", "%d/%m/%Y", "%m/%d/%Y", "%b %d, %Y"]
MESES_CIERRE = {"Walmart": {1, 2}, "Target": {1, 2}, "Costco": {8, 9}}

def leer_fecha(texto, empresa):
    candidatas = set()
    for f in FORMATOS:
        try:
            candidatas.add(pd.to_datetime(texto, format=f))
        except ValueError:
            pass
    if len(candidatas) == 1:
        return candidatas.pop(), None
    plausibles = [c for c in candidatas if c.month in MESES_CIERRE[empresa]]
    if len(plausibles) != 1:
        raise ValueError(f"Fecha imposible de resolver: {texto!r} ({empresa})")
    return plausibles[0], "fecha ambigua resuelta por calendario fiscal"

fechas = []
for i, (t, e) in df[["cierre_fiscal", "empresa"]].iterrows():
    fecha, nota = leer_fecha(t.strip(), e)
    if nota or t != fecha.strftime("%Y-%m-%d"):
        anotar(i, "cierre_fiscal", t, fecha.strftime("%Y-%m-%d"), nota or "formato ISO")
    fechas.append(fecha)
df["cierre_fiscal"] = fechas

**Montos.** Conviven `$680,985` (coma de miles), `242.290` (punto de miles, a la
española), `611289.0` (punto decimal) y celdas vacías marcadas como `N/A` o `-`. La regla
para el punto: si separa grupos de exactamente tres dígitos, es separador de miles. El
supuesto funciona porque las cifras están en millones y ninguna tiene decimales con tres
dígitos; con otros datos habría que revisarlo.

In [7]:
FALTANTES = {"", "n/a", "na", "-", "n.d.", "nd"}

def leer_monto(texto):
    t = texto.strip().replace("$", "").replace(" ", "")
    if t.lower() in FALTANTES:
        return np.nan
    if re.fullmatch(r"\d{1,3}(\.\d{3})+", t):      # 242.290 -> miles a la española
        return float(t.replace(".", ""))
    return float(t.replace(",", ""))                # $680,985 o 611289.0

for c in CUENTAS:
    nuevos = df[c].map(leer_monto)
    for i in df.index:
        if pd.isna(nuevos[i]):
            anotar(i, c, df.at[i, c], "NaN", "faltante: se deja vacío, no se imputa")
        elif df.at[i, c] != str(int(nuevos[i])):
            anotar(i, c, df.at[i, c], nuevos[i], "texto a número")
    df[c] = nuevos

monedas = df["moneda"].str.strip().str.upper().replace({"US$": "USD", "DÓLARES": "USD"})
assert set(monedas) == {"USD"}, f"Monedas inesperadas: {set(monedas)}"
df["moneda"] = monedas

**Duplicados.** Después de normalizar nombres y fechas aparecen los duplicados que antes
estaban escondidos detrás de formatos distintos. Si dos filas de la misma empresa y
cierre tuvieran montos distintos, no se elegiría una al azar: el código se detendría.

In [8]:
#| label: out-dedup
claves = ["empresa", "cierre_fiscal"]
grupos = df.groupby(claves)[CUENTAS].nunique(dropna=False)
conflictos = grupos[(grupos > 1).any(axis=1)]
assert conflictos.empty, f"Duplicados con montos distintos:\n{conflictos}"

repetidas = df[df.duplicated(claves)].index
for i in repetidas:
    anotar(i, "fila completa", "", "eliminada", "duplicado de empresa y cierre")
df = df.drop(index=repetidas)
print(f"Duplicados eliminados: {len(repetidas)}. Quedan {len(df)} filas.")

Duplicados eliminados: 2. Quedan 15 filas.


**Escala.** Los dos errores más peligrosos no son de formato: un monto cargado en miles de
millones en lugar de millones, y un cero de más. Ambos producen números válidos. Se
detectan comparando cada valor con la mediana de esa cuenta para la misma empresa, y se
corrigen solo si la corrección es inequívoca: un factor de 1.000 (unidad equivocada) o
un factor de 10 confirmado por la identidad contable activo ≈ pasivo + patrimonio.

In [9]:
#| label: out-escala
df = df.sort_values(claves).reset_index().rename(columns={"index": "fila"})
for c in CUENTAS:
    mediana = df.groupby("empresa")[c].transform("median")
    razon = df[c] / mediana
    for i in df.index[razon < 1 / 200]:
        nuevo = df.at[i, c] * 1000
        anotar(df.at[i, "fila"], c, df.at[i, c], nuevo, "unidad corregida; la precisión perdida no se recupera: verificar en la fuente")
        print(f"{df.at[i, 'empresa']} {df.at[i, 'cierre_fiscal']:%Y-%m} {c}: {df.at[i, c]} -> {nuevo:,.0f}")
        df.at[i, c] = nuevo
    for i in df.index[(razon > 5) & (razon < 20)]:
        candidato = df.at[i, c] / 10
        if c == "activos":
            esperado = df.at[i, "pasivo_total"] + df.at[i, "patrimonio"]
            if abs(candidato / esperado - 1) < 0.05:
                anotar(df.at[i, "fila"], c, df.at[i, c], candidato, "cero de más, confirmado por identidad contable")
                print(f"{df.at[i, 'empresa']} {df.at[i, 'cierre_fiscal']:%Y-%m} {c}: {df.at[i, c]:,.0f} -> {candidato:,.0f}")
                df.at[i, c] = candidato
                continue
        print(f"REVISAR A MANO: {df.at[i, 'empresa']} {df.at[i, 'cierre_fiscal']:%Y-%m} {c} = {df.at[i, c]:,.0f}")

Walmart 2024-01 ingresos: 648.1 -> 648,100
Costco 2024-09 activos: 698,310 -> 69,831


## 3. Validación contra la verdad

Como el archivo sucio se creó a partir de los datos del día 1, hay una clave de
respuestas: se puede comparar cada celda limpia con el valor original. En un caso real
no existe esa clave, y la validación se hace con controles como la identidad contable o
el cotejo contra el 10-K que vimos el día 1.

In [10]:
#| label: tbl-validacion
ref = pd.read_csv(DATA / "referencia_dia1.csv", parse_dates=["cierre_fiscal"])
ref[CUENTAS] = (ref[CUENTAS] / 1e6).round()
comp = df.merge(ref[claves + CUENTAS], on=claves, suffixes=("", "_ref"), how="outer", indicator=True)
assert (comp["_merge"] == "both").all(), "Hay filas que no coinciden con la referencia"

resumen = []
for c in CUENTAS:
    iguales = (comp[c] == comp[f"{c}_ref"]).sum()
    vacias = comp[c].isna().sum()
    resumen.append({"Cuenta": c, "Celdas iguales a la referencia": f"{iguales} de {len(comp)}",
                    "Vacías (faltantes no imputados)": vacias,
                    "Distintas": len(comp) - iguales - vacias})
pd.DataFrame(resumen).style.hide(axis="index")

Cuenta,Celdas iguales a la referencia,Vacías (faltantes no imputados),Distintas
ingresos,14 de 15,0,1
utilidad_neta,15 de 15,0,0
activos,15 de 15,0,0
patrimonio,15 de 15,0,0
activo_corriente,14 de 15,1,0
pasivo_corriente,14 de 15,1,0
pasivo_total,14 de 15,1,0


Quedan cuatro celdas sin coincidir. Una es el ingreso de Walmart que venía en miles de
millones: multiplicar por 1.000 corrige el orden de magnitud, pero los dígitos que se
perdieron al redondear a 648,1 no vuelven. La cifra limpia queda 25 millones por debajo
de la real, y por eso el registro marca esa corrección para verificarla en la fuente.

Las otras tres son los faltantes, que se dejaron vacíos a propósito. ¿Por qué no imputarlos? Para el pasivo total de Walmart sería tentador usar
la identidad contable, activo menos patrimonio. La celda siguiente muestra cuánto se
equivocaría esa imputación.

In [11]:
#| label: out-imputacion
w = comp[(comp["empresa"] == "Walmart") & comp["pasivo_total"].isna()].iloc[0]
imputado = w["activos"] - w["patrimonio"]
print(f"Walmart, cierre {w['cierre_fiscal']:%Y-%m}")
print(f"  Pasivo imputado (activo - patrimonio): {imputado:,.0f} millones")
print(f"  Pasivo real según el 10-K:              {w['pasivo_total_ref']:,.0f} millones")
print(f"  Diferencia:                             {imputado - w['pasivo_total_ref']:,.0f} millones")

Walmart, cierre 2023-01
  Pasivo imputado (activo - patrimonio): 166,504 millones
  Pasivo real según el 10-K:              159,443 millones
  Diferencia:                             7,061 millones


La diferencia es la **participación no controladora**: la parte del patrimonio de las
filiales que pertenece a otros accionistas. La columna `patrimonio` es solo el de la
controladora, así que la identidad no cierra con ella. Una imputación "lógica" introduce
un error de miles de millones sin dar ninguna alerta.

In [12]:
#| label: tbl-sembrados
sembrados = pd.read_csv(DATA / "errores_sembrados.csv")
pd.DataFrame({"Errores sembrados": sembrados["tipo"].value_counts()}).rename_axis("Tipo")

,Errores sembrados
Tipo,
número como texto,77
nombre inconsistente,12
formato de fecha,12
moneda inconsistente,9
valor faltante,3
unidad equivocada (miles de millones),1
cero de más,1
duplicado exacto,1
duplicado con otro formato,1


## 4. Guardar el resultado y el registro

`estados_limpio.csv` queda listo para el dashboard del día 5. `registro_cambios.csv`
documenta cada modificación con su regla: es lo que permite a otra persona auditar la
limpieza sin repetirla.

In [13]:
#| label: out-guardar
df["margen_neto"] = df["utilidad_neta"] / df["ingresos"]
df["razon_corriente"] = df["activo_corriente"] / df["pasivo_corriente"]
salida = df.drop(columns=["fila", "margen_reportado"])
salida.to_csv(DATA / "estados_limpio.csv", index=False, date_format="%Y-%m-%d")

reg = pd.DataFrame(registro)
reg.to_csv(DATA / "registro_cambios.csv", index=False)
print(f"{len(salida)} filas limpias. {len(reg)} cambios registrados:")
print(reg["regla"].value_counts().to_string())

15 filas limpias. 115 cambios registrados:
regla
texto a número                                                                   81
nombre canónico                                                                  13
formato ISO                                                                      12
faltante: se deja vacío, no se imputa                                             3
fecha ambigua resuelta por calendario fiscal                                      2
duplicado de empresa y cierre                                                     2
unidad corregida; la precisión perdida no se recupera: verificar en la fuente     1
cero de más, confirmado por identidad contable                                    1
